In [8]:
#@title Simulador de Produção Solar ☀️

# =====================================================
# IMPORTS
# =====================================================

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime
from scipy.interpolate import interp1d

import ipywidgets as widgets
from IPython.display import display, clear_output

# =====================================================
# CONFIGURAÇÕES DA USINA
# =====================================================

POTENCIA_USINA = 112          # kWp
PERFORMANCE_RATIO = 0.75

LATITUDE = -22.8727
LONGITUDE = -42.3436

# =====================================================
# CÓDIGOS WMO
# =====================================================

WMO = {

0:"☀️ Céu limpo",
1:"🌤 Predominantemente limpo",
2:"⛅ Parcialmente nublado",
3:"☁️ Nublado",

45:"🌫 Neblina",
48:"🌫 Neblina congelante",

51:"🌦 Garoa fraca",
53:"🌦 Garoa moderada",
55:"🌦 Garoa intensa",

61:"🌧 Chuva fraca",
63:"🌧 Chuva moderada",
65:"🌧 Chuva forte",

71:"❄️ Neve fraca",
73:"❄️ Neve moderada",
75:"❄️ Neve intensa",

80:"🌦 Pancadas fracas",
81:"🌦 Pancadas moderadas",
82:"⛈ Pancadas fortes",

95:"⛈ Tempestade",
96:"⛈ Tempestade com granizo",
99:"⛈ Tempestade severa"

}

# =====================================================
# CONSULTA OPEN METEO
# =====================================================

def consultar_api(dia):

    hoje = datetime.now()

    data = f"{hoje.year}-{hoje.month:02d}-{dia:02d}"

    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={LATITUDE}"
        f"&longitude={LONGITUDE}"
        f"&start_date={data}"
        f"&end_date={data}"
        "&hourly=shortwave_radiation,cloud_cover,weather_code"
        "&timezone=America/Sao_Paulo"
    )

    resposta = requests.get(url)

    dados = resposta.json()["hourly"]

    df = pd.DataFrame({

        "Hora":dados["time"],
        "Irradiacao":dados["shortwave_radiation"],
        "Nebulosidade":dados["cloud_cover"],
        "WMO":dados["weather_code"]

    })

    return df

    # =====================================================
# INTERPOLAÇÃO PARA 5 MINUTOS
# =====================================================

def interpolar(df):

    # Converte a coluna de horário
    df["Hora"] = pd.to_datetime(df["Hora"])

    # Hora decimal (0 a 23)
    horas = (
        df["Hora"].dt.hour +
        df["Hora"].dt.minute/60
    )

    # Novo eixo de 5 em 5 minutos
    novo_eixo = np.arange(5, 21.01, 5/60)

    f_irr = interp1d(
        horas,
        df["Irradiacao"],
        kind="cubic"
    )

    f_nuvem = interp1d(
        horas,
        df["Nebulosidade"],
        kind="linear"
    )

    irr = f_irr(novo_eixo)
    nuvem = f_nuvem(novo_eixo)

    clima = []

    for h in novo_eixo:

        indice = np.argmin(np.abs(horas - h))

        clima.append(df.iloc[indice]["WMO"])

    resultado = pd.DataFrame({

        "HoraDecimal": novo_eixo,
        "Hora": [
            f"{int(h):02d}:{int(round((h%1)*60)):02d}"
            for h in novo_eixo
        ],
        "Irradiacao": irr,
        "Nebulosidade": nuvem,
        "WMO": clima

    })

    return resultado

    # =====================================================
# FORMULÁRIO
# =====================================================

dia = widgets.BoundedIntText(

    value=datetime.now().day,

    min=1,

    max=31,

    description="Dia"

)

botao = widgets.Button(

    description="Simular",

    button_style="warning"

)

saida = widgets.Output()

display(dia)
display(botao)
display(saida)

# =====================================================
# EVENTO DO BOTÃO
# =====================================================

def executar(b):

    with saida:

        clear_output()

        bruto = consultar_api(dia.value)

        interpolado = interpolar(bruto)

        display(interpolado.head())

botao.on_click(executar)

# =====================================================
# CÁLCULO DA PRODUÇÃO
# =====================================================

def calcular_producao(df):

    df = df.copy()

    # Ajuste leve pela nebulosidade
    fator_nuvem = 1 - (df["Nebulosidade"] / 100) * 0.20

    df["Potencia_kW"] = (
        POTENCIA_USINA *
        (df["Irradiacao"] / 1000) *
        PERFORMANCE_RATIO *
        fator_nuvem
    )

    df["Potencia_kW"] = df["Potencia_kW"].clip(lower=0)

    energia = (df["Potencia_kW"] * (5/60)).sum()

    pico = df["Potencia_kW"].max()

    horario_pico = df.loc[
        df["Potencia_kW"].idxmax(),
        "Hora"
    ]

    df['Potencia_kW'] = df['Potencia_kW'].round(2)
    df["Clima"] = df["WMO"].map(WMO)

    return df, energia, pico, horario_pico

    # =====================================================
# GRÁFICO
# =====================================================

def grafico(df):

    plt.figure(figsize=(10,5))

    plt.plot(
        df["Hora"],
        df["Potencia_kW"],
        color="orange",
        linewidth=3
    )

    plt.fill_between(
        df["Hora"],
        df["Potencia_kW"],
        color="orange",
        alpha=.35
    )

    plt.xticks(df.index[::12], rotation=45)

    plt.ylabel("Produção (kW)")
    plt.xlabel("Hora")

    plt.title("Produção Estimada da Usina")

    plt.grid(alpha=.30)

    plt.tight_layout()

    plt.show()

    # =====================================================
# EXECUÇÃO
# =====================================================

def executar(b):

    with saida:

        clear_output()

        bruto = consultar_api(dia.value)

        interpolado = interpolar(bruto)

        resultado, energia, pico, horario = calcular_producao(interpolado)

        pd.set_option('display.float_format', lambda x: '%.2f' % x)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)

        print("="*50)
        print("SIMULAÇÃO DE PRODUÇÃO SOLAR")
        print("="*50)
        print(f"Potência da usina : {POTENCIA_USINA:.2f} kWp")
        print(f"Performance Ratio : {PERFORMANCE_RATIO*100:.0f}%")
        print()
        print(f"Energia estimada : {energia:.2f} kWh")
        print(f"Pico de potência : {pico:.2f} kW")
        print(f"Horário do pico  : {horario}")
        print("="*50)

        grafico(resultado)

        display(

            resultado[[
                "Hora",
                "Irradiacao",
                "Nebulosidade",
                "Clima",
                "Potencia_kW"
            ]]

        )

botao.on_click(executar)

BoundedIntText(value=30, description='Dia', max=31, min=1)

Button(button_style='warning', description='Simular', style=ButtonStyle())

Output()